**Name:** *Muhammad Bilal*

**Department:** *Data Science*

**University:** *Ghulam Ishaq Khan Institute of Engineering and Sciences*

# **Decode Labs Internship**

I used a Python Notebook since that is the medium I am most familiar with in terms of working with datasets and the tasks that need to be performed in the projects.

### **Data Science Project 1:** Advanced EDA & Feature Engineering
**Objective:** Transform raw, chaotic data into a mathematically clean dataset ready for machine learning algorithms.

### **Task 1:** Data Collection & Loading
**Goal:** Collect or load a dataset

In [7]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load the dataset
print("Loading data...")
df = pd.read_csv('mymoviedb.csv', sep=None, engine='python', on_bad_lines='skip')
df.head(3)

Loading data...


,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...


### **Task 2:** Data Cleaning & Preprocessing
**Goal:**
Prepare the dataset for analysis by cleaning and organizing data.
Handling missing values using statistical imputation rather than arbitrary guesswork, and dropping columns with severe missingness (>20%).

In [9]:
# Identify proportion of missingness per feature
missing_pct = df.isnull().mean()

# Drop features with > 20% missing data
cols_to_drop = missing_pct[missing_pct > 0.20].index
df.drop(columns=cols_to_drop, inplace=True)

# Define numerical columns for Global Median Imputation
num_cols = ['Popularity', 'Vote_Count', 'Vote_Average']
num_cols = [c for c in num_cols if c in df.columns]

# --- THE FIX: Force numeric data types ---
# This converts any stray strings (like 'en') into NaN so the imputer can handle them.
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# -----------------------------------------

# Now the imputer will work perfectly
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])

# Sub-Group Conditional Imputation for Categoricals
cat_cols = ['Overview', 'Genre', 'Original_Language']
cat_cols = [c for c in cat_cols if c in df.columns]
df[cat_cols] = df[cat_cols].fillna('Unknown')

print("Missing values handled and data types enforced.")

Missing values handled and data types enforced.


### **Task 3:** Neutralizing Outliers
**Goal:** Using the Interquartile Range (IQR) to establish mathematical boundaries and cap extreme values, protecting the distribution variance.

In [10]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap outliers
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])

print("Outliers neutralized via IQR boundaries.")

Outliers neutralized via IQR boundaries.


### **Task 4:** Feature Engineering
**Goal:** Extracting three new predictive features
1. `Release_Year` (Temporal Extraction)
2. `Engagement_Score` (Optimization Target)
3. `Genre_Count` (Complexity Index)

In [11]:
# Feature 1: Temporal Extraction
if 'Release_Date' in df.columns:
    df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
    df['Release_Year'] = df['Release_Date'].dt.year
    df['Release_Year'] = df['Release_Year'].fillna(df['Release_Year'].median())

# Feature 2: Optimization Target
if 'Popularity' in df.columns and 'Vote_Average' in df.columns:
    df['Engagement_Score'] = df['Popularity'] * df['Vote_Average']

# Feature 3: Complexity Index
if 'Genre' in df.columns:
    df['Genre_Count'] = df['Genre'].apply(lambda x: len(str(x).split(',')) if x != 'Unknown' else 0)

# Verify the final structure
print("Pipeline complete. Displaying final structured dataset:\n")
df[['Title', 'Release_Year', 'Engagement_Score', 'Genre_Count']].head()

Pipeline complete. Displaying final structured dataset:



,Title,Release_Year,Engagement_Score,Genre_Count
0,Spider-Man: No Way Home,2021.0,528.7515,3
1,The Batman,2022.0,516.0105,3
2,No Exit,2022.0,401.3415,1
3,Encanto,2021.0,490.5285,4
4,The King's Man,2021.0,445.9350,4


### **Final Thoughts (What I took away from this project):**


1. **Data Integrity over Deletion:** I learned that dropping rows with missing data or outliers destroys valuable information. Using median imputation and IQR capping protects the variance and structure of the dataset.

2. **The Power of Feature Engineering:** Taking raw text and dates and turning them into mathematical signals (like a Genre_Count complexity index or Release_Year) taught me how to make data actually useful for algorithms.

3. **Production-Ready Code:** I moved away from messy, one-off commands and learned how to structure an Input-Process-Output (IPO) pipeline in a notebook, making my work scalable and professional.